In [47]:
"""
PM2.5 Data Processing Pipeline
===============================
This notebook combines SD card data and app download data, merges with existing master data,
and applies cleaning and calibration corrections.

Usage:
1. Set paths in the CONFIG section
2. Run all cells
3. Output will be saved to master CSV and a timestamped backup
"""

import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
import shutil
from collections import defaultdict


# ============================================================================
# CONFIGURATION - SET YOUR PATHS HERE
# ============================================================================

CONFIG = {
    # Input paths - ADD ALL YOUR FOLDERS HERE
    'sd_card_folders':  [
        "/Users/aatwood/Downloads/Sensores dados 2024-2025",
        "/Users/aatwood/Downloads/Sensores dados 2023-2024"
        # Add more SD card folders here None,
    ],
    
    'download_folders': [
        "/Users/aatwood/Documents/downloaded_purple_air_dataJune1_2025",
        "/Users/aatwood/Documents/PurpleAir Download 9-30-2025(1)",
        "/Users/aatwood/Documents/PurpleAir Download 6-26-2025",
        "/Users/aatwood/Documents/PurpleAir Download 9-30-2025",
        "/Users/aatwood/Documents/PurpleAir Download 12-3-2025",
    "/Users/aatwood/Downloads/PurpleAir Download 1-5-2026",
        "/Users/aatwood/Downloads/PurpleAir Download 1-5-2026(1)"
        
        
        # Add more download folders here None, #
    ],
    
    # External CSV file (para_air or similar)
    #'external_csv': "/Users/aatwood/Downloads/acre_PA_data/tb_parealtimedata.csv",  # Set to None if not using
    
    'reference_file': "/Users/aatwood/Documents/WW_IPAM_PA_sensors_June2025.csv",
    
    # Output paths
    'master_csv': "/Users/aatwood/Documents/PM25_master_data.csv", # "/Users/aatwood/Documents/PM25_master_data.csv",
    'backup_folder': "/Users/aatwood/Documents/PM25_backups",
    
    # Reference file columns
    'code_column': "Code",
    'site_column': "Name",
    
    # Data quality thresholds
    'min_temp': 30,
    'max_temp': 150,
    'min_humidity': 0,
    'max_humidity': 100,
    'max_abs_diff': 5,
    'max_percent_diff': 33
}


## WW/IPAM Purple Air site names
site_name_map = {
    'Alta Floresta': 'Alta Floresta URB',
    'Alta Foresta URB 2024 2025': 'Alta Floresta URB',
    'Ar Alta Foresta URB 2024 2025': 'Alta Floresta URB',
    'Ar Alta Floresta URB': 'Alta Floresta URB',
    'Altamira 1': 'Altamira URB',
    'Altamira 2': 'Altamira URB',
    'Ar Altamira URB': 'Altamira URB',
    'Ar Altamira URB 2024 2025': 'Altamira URB',
    'Altamira URB 2024 2025': 'Altamira URB',
    'Alto Turiaçu': 'Alto Turiaçu TI',
    'AR Alto Turiacu TI': 'Alto Turiaçu TI',
    'IPAM Canarana Office': 'Canarana URB',
    'Ar Canarana URB':'Canarana URB',
    'Capitao Poco URB': 'Capitão Poço URB',
    'UFRA Capitão Poço 2': 'Capitão Poço URB',
    'UFRA Capitão Poço 1': 'Capitão Poço URB',
    'Ar Capitao Poco URB': 'Capitão Poço URB',
    'UFRA/Capitão Poço 1': 'Capitão Poço URB',
    'Chapada dos Guimaraes UC': 'Chapada dos Guimarães UC',
    'Ar Chapada dos Guimaraes UC': 'Chapada dos Guimarães UC',
    'Chapada dos Guimarães': 'Chapada dos Guimarães UC',
    'Caru': 'Caru TI',
    'Cuniã': 'Cuniã UC',
    'Cunia UC': 'Cuniã UC',
    'Ar Cunia UC': 'Cuniã UC',
      'itamarati URB': 'Itamarati URB',
    'Itamarati': 'Itamarati URB',
        'Ar itamarati URB': 'Itamarati URB',
    'Ar Itamarati': 'Itamarati URB',
    'Ar Itamarati URB': 'Itamarati URB',
    'Mamiraua': 'Mamiraua UC',
    'AR Nove de Janeiro TI': 'Nove de Janeiro TI',
    'Nove de Janeiro': 'Nove de Janeiro TI',
    'AR ONF Fazenda': 'ONF Fazenda',
        'Ar ONF Fazenda': 'ONF Fazenda',
    'AR Parque do araguaia TI': 'Parque do Araguaia TI',
    'Parque do aguaia 2': 'Parque do Araguaia TI',
    'Parque do Araguaia 1': 'Parque do Araguaia TI',
    'Katukina/ Kaxinawá':'Katukina_Kaxinawá TI',
    'Santarém Baixar': 'Santarém URB',
    'Santarem URB': 'Santarém URB',
    'Ar Santarem URB': 'Santarém URB',
    'IPAM Santarem': 'Santarém URB',
    'Sinop': 'Sinop URB',
    'Ar Sinop': 'Sinop URB',
    'Ar Sinop URB': 'Sinop URB',
    'Tanguro Fazenda': 'Tanguro Fazenda',
    'PELD TANG 1': 'Tanguro Fazenda',
    'PELD-TANG 1': 'Tanguro Fazenda',
    'PELD TANG 2': 'Tanguro Fazenda',
    'Tefe': 'Tefe URB',
    'Ar Tefe URB': 'Tefe URB',
    'Uaca TI': 'Uaça TI',
    'Uaça':'Uaça TI',
     'Ar Uaca TI': 'Uaça TI',
    'Ar Uaça':'Uaça TI',
     'AR Xerentes Tocantins TI': 'Xerentes Tocantins TI',
    'Xerente 1': 'Xerentes Tocantins TI',
    'Xerente 2': 'Xerentes Tocantins TI',
    'Fazenda São Nicolau   ONF Brasil Gestão Florestal': 'Fazenda São Nicolau',
    'Fazenda São Nicolau - ONF Brasil Gestão Florestal': 'Fazenda São Nicolau',
    'Fazenda São Nicolau-ONF Brasil Gestão Florestal': 'Fazenda São Nicolau',
    'Fazenda São Nicolau ONF Brasil Gestão Florestal': 'Fazenda São Nicolau',
    'ONF Fazenda': 'Fazenda São Nicolau',
    'GUAJARA MIRIM': 'Guajara Mirim',
    'GUAJARA-MIRIM': 'Guajara Mirim',
    'São Félix do Xingu 1': 'São Félix do Xingu URB',
      'São Félix do Xingu 2': 'São Félix do Xingu URB',
    'São Felix  do Xingu URB': 'São Félix do Xingu URB',
    'Ar São Felix  do Xingu URB ': 'São Félix do Xingu URB',
    'ariboia': 'Ariboia TI',
    'Mamiraua': 'Mamiraua UC',
    'Ar Mamiraua': 'Mamiraua UC',
    'Ar Mamiraua UC': 'Mamiraua UC',
    'IPAM Cuiabá': 'Cuiabá URB',
    'Ar Cuiabá URB': 'Cuiabá URB',
    'Serra da Moça 1': 'Serra da Moça TI',
    'Serra da Moça 2': 'Serra da Moça TI',    
    'Serra da Moça': 'Serra da Moça TI',
    'Boca da mata': 'Boca da Mata',
    'TI Raposa Serra do Sol': 'Raposa Serra do Sol TI',
    'TI Vista Alegre': 'Vista Alegre TI',
    'Juina': 'Juina URB',
    'Ar Juina': 'Juina URB',
    'Ar Juina URB': 'Juina URB',
    'Ar Tanguro Fazenda': 'Tanguro Fazenda',
    'Tanguro Fazenda': 'Tanguro Fazenda',
    'PELD TANG 1': 'Tanguro Fazenda',
    'PELD-TANG 1 ': 'Tanguro Fazenda',
    'PELD TANG 2': 'Tanguro Fazenda',
    'Tefe': 'Tefe URB',
    'Uaca TI': 'Uaça TI',
    'Uaça TI ': 'Uaça TI',
    'Uaça TI': 'Uaça TI',
    'Uaça':'Uaça TI',
    'Uaça':'Uaça TI',
    'Ar Uaça':'Uaça TI',
    'Ar Uaça TI':'Uaça TI',
    'Mãe Maria': 'Mãe Maria TI',
    'Merure': 'Merure TI',
    'Nove de Janeiro': 'Nove de Janeiro TI',
    'Vale do Javari': 'Vale do Javari TI',
    'Uru-Eu-Wau-Wau': 'Uru-Eu-Wau-Wau TI',
    'Kadiwéu': 'Kadiwéu TI',
    'Jofre Velho': 'Jofre Velho Fazenda',
    'Instituto Kabu 1': 'Instituto Kabu',
    'Ar_Uaça_TI' : 'Uaça TI', 
    'Ar_Juina_URB': 'Juina URB', 
    'Ar_Cuiabá_URB': 'Cuiabá URB',
       'Ar_São_Felix_ do_Xingu_URB_': 'São Félix do Xingu URB' , 
    'Ar_Alta_Foresta_URB_2024-2025': 'Alta Floresta URB',
       'Ar-Itamarati_URB':'Itamarati URB', 
    'Ar_Sinop_URB': "Sinop URB", 
    'Ar_Altamira_URB_2024-2025': 'Altamira URB',
    'Ar_ONF_Fazenda' : 'Fazenda São Nicolau', 
    'Ar_Alta_Floresta_URB':'Alta Floresta URB',
       'Ar_Altamira_URB':'Altamira URB', 
    'AR_Alto_Turiacu_TI':'Alto Turiaçu TI', 
    'Ar_Canarana_URB':'Canarana URB',
       'Ar_Capitao_Poco_URB':'Capitão Poço URB',
    'Ar_Chapada_dos_Guimaraes_UC':'Chapada dos Guimarães UC',
       'Ar_Cunia_UC':'Cuniã UC', 
    'Ar_itamarati_URB':'Itamarati URB', 
    'Ar_Mamiraua_UC': 'Mamiraua UC',
       'AR_Nove_de_Janeiro_TI': 'Nove de Janeiro TI', 
    'AR_ONF_Fazenda':'Fazenda São Nicolau',
       'AR_Parque_do_araguaia_TI':'Parque do Araguaia TI', 
    'Ar_Santarem_URB': 'Santarém URB',
       'Ar_Tanguro_Fazenda':'Tanguro Fazenda', 
    'Ar_Tefe_URB': 'Tefe URB', 
    'Ar_Uaca_TI': 'Uaça TI',
       'AR_Xerentes_Tocantins_TI': 'Xerentes Tocantins TI',
    
    # Add more mappings as needed
}


In [48]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def create_backup(master_csv_path, backup_folder):
    """Create a timestamped backup of the master CSV"""
    if os.path.exists(master_csv_path):
        os.makedirs(backup_folder, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = os.path.join(backup_folder, f"PM25_master_backup_{timestamp}.csv")
        shutil.copy2(master_csv_path, backup_path)
        print(f"✓ Backup created: {backup_path}")
        return backup_path
    else:
        print("ℹ No existing master CSV found - will create new one")
        return None


def load_master_data(master_csv_path):
    """Load existing master CSV or return empty DataFrame"""
    if os.path.exists(master_csv_path):
        df = pd.read_csv(master_csv_path)
        df['time_bsb_copy'] = pd.to_datetime(df['time_bsb_copy'], utc=True)
        print(f"✓ Loaded master data: {len(df)} rows")
        return df
    else:
        print("ℹ No master data found - starting fresh")
        return pd.DataFrame()

In [49]:
def process_all_download_data(download_folders, reference_file_path, code_column, site_column):
    """
    Process download data that may be split across multiple CSV files and folders.
    Groups files by sensor code, merges/concatenates as needed, and maps to sites.
    """
    print("\n" + "="*70)
    print("PROCESSING DOWNLOAD DATA (WITH CROSS-FOLDER MERGE)")
    print("="*70)
    
    # Load reference data
    if not os.path.exists(reference_file_path):
        print(f"❌ Reference file not found: {reference_file_path}")
        return pd.DataFrame()
    
    ref_df = pd.read_csv(reference_file_path)
    print(f"✓ Reference file loaded: {len(ref_df)} rows")
    # Handle None or empty list
    if not download_folders:
        print("⚠ No download folders configured - skipping")
        return pd.DataFrame()
    # Create code-to-site mapping
    code_to_site = {}
    
    # Clean reference data
    ref_df_clean = ref_df.copy()
    for col in [code_column, 'NewCode']:
        if col in ref_df_clean.columns:
            ref_df_clean[col] = pd.to_numeric(ref_df_clean[col], errors='coerce')
            ref_df_clean[col] = ref_df_clean[col].astype('Int64').astype(str)
            ref_df_clean[col] = ref_df_clean[col].replace(['nan', '<NA>'], pd.NA)
            if ref_df_clean[col].dtype == 'object':
                ref_df_clean[col] = ref_df_clean[col].str.strip()
    
    # Build mapping from both code columns
    if code_column in ref_df_clean.columns:
        mapping = ref_df_clean.dropna(subset=[code_column]).set_index(code_column)[site_column].to_dict()
        code_to_site.update(mapping)
    
    if 'NewCode' in ref_df_clean.columns:
        mapping = ref_df_clean.dropna(subset=['NewCode']).set_index('NewCode')[site_column].to_dict()
        code_to_site.update(mapping)
    
    print(f"✓ Created {len(code_to_site)} code-to-site mappings")
    
    # STEP 1: Collect all files from all folders, grouped by sensor code
    print("\n--- Step 1: Scanning all folders ---")
    files_by_sensor = defaultdict(list)
    
    for folder in download_folders:

        if not os.path.exists(folder):
            print(f"⚠️  Folder not found: {folder}")
            continue
        
        print(f"Scanning: {folder}")
        csv_files = [f for f in os.listdir(folder) if f.endswith('.csv')]
        print(f"  Found {len(csv_files)} CSV files")
        
        for file in csv_files:
            # Extract sensor code from filename (assumes format like "128391 2023-01-01...")
            parts = file.split()
            if parts:
                sensor_code = parts[0]
                file_path = os.path.join(folder, file)
                files_by_sensor[sensor_code].append(file_path)
    
    print(f"\n✓ Found {len(files_by_sensor)} unique sensors across all folders")
    
    # STEP 2: Process each sensor's files
    all_data = []
    
    for sensor_code, file_paths in files_by_sensor.items():
        print(f"\n{'='*70}")
        print(f"Sensor {sensor_code}: {len(file_paths)} file(s) across folders")
        print(f"{'='*70}")
        
        # Load all files for this sensor
        sensor_dfs = []
        for file_path in file_paths:
            file_name = os.path.basename(file_path)
            try:
                df = pd.read_csv(file_path, low_memory=False)
                print(f"  ✓ {file_name}")
                print(f"    Rows: {len(df)}, Columns: {len(df.columns)}")
                
                # Ensure time_stamp exists
                if 'time_stamp' not in df.columns:
                    print(f"    ⚠️  No time_stamp column, skipping")
                    continue
                
                # Convert and validate timestamps
                df['time_stamp'] = pd.to_datetime(df['time_stamp'], errors='coerce')
                valid_timestamps = df['time_stamp'].notna().sum()
                
                if valid_timestamps == 0:
                    print(f"    ⚠️  All timestamps invalid, skipping")
                    continue
                
                # Filter to valid timestamps only
                df = df[df['time_stamp'].notna()]
                
                # Show timestamp range
                print(f"    Time range: {df['time_stamp'].min()} to {df['time_stamp'].max()}")
                print(f"    Valid timestamps: {len(df)}")
                
                # Show key columns present
                key_cols = ['temperature', 'humidity', 'pm2.5_cf_1_a', 'pm2.5_cf_1_b', 'pm2.5_cf_1']
                present = [col for col in key_cols if col in df.columns]
                if present:
                    print(f"    Key columns: {present}")
                
                # CRITICAL: Skip files that are missing essential sensor data columns
                # These are incomplete downloads with only metadata columns
                essential_cols = ['temperature', 'humidity']
                missing_essential = [col for col in essential_cols if col not in df.columns]
                
                if missing_essential:
                    print(f"    ⚠️  SKIPPING - Missing essential columns: {missing_essential}")
                    print(f"    (This file has metadata but no sensor data)")
                    continue
                
                # Also skip if columns exist but have no data
                has_data = all(df[col].notna().sum() > 0 for col in essential_cols if col in df.columns)
                if not has_data:
                    print(f"    ⚠️  SKIPPING - Essential columns are all null")
                    continue
                
                sensor_dfs.append(df)
                    
            except Exception as e:
                print(f"    ❌ Error loading {file_name}: {e}")
        
        if not sensor_dfs:
            print(f"  ⚠️  No valid data files for sensor {sensor_code}")
            continue
        
        # STEP 3: Check if files need merging or just concatenation
        print(f"\n  --- Processing {len(sensor_dfs)} file(s) ---")
        
        if len(sensor_dfs) == 1:
            merged = sensor_dfs[0]
            print(f"  → Single file, no merge needed: {len(merged)} rows")
        else:
            # Check if files have overlapping columns (need merge) or complementary columns (need concat)
            key_cols = ['temperature', 'humidity', 'pm2.5_cf_1_a', 'pm2.5_cf_1_b']
            
            # Check if first file has all key columns
            first_has_all = all(col in sensor_dfs[0].columns for col in key_cols)
            
            # Check timestamp overlap
            time_ranges = [(df['time_stamp'].min(), df['time_stamp'].max()) for df in sensor_dfs]
            print(f"  Time ranges:")
            for i, (start, end) in enumerate(time_ranges):
                print(f"    File {i+1}: {start} to {end}")
            
            # If files have different date ranges and first has all columns, concatenate instead of merge
            overlapping_ranges = False
            for i in range(len(time_ranges) - 1):
                for j in range(i + 1, len(time_ranges)):
                    if time_ranges[i][1] >= time_ranges[j][0] and time_ranges[j][1] >= time_ranges[i][0]:
                        overlapping_ranges = True
                        break
            
            if first_has_all and not overlapping_ranges:
                # Files are time-sequential with all columns - just concatenate
                print(f"  → Files are sequential, concatenating instead of merging")
                merged = pd.concat(sensor_dfs, ignore_index=True)
                merged = merged.sort_values('time_stamp').reset_index(drop=True)
                print(f"  Result: {len(merged)} rows, {len(merged.columns)} columns")
            else:
                # Files have overlapping times or missing columns - need to merge
                print(f"  → Files need merging (overlapping times or complementary columns)")
                merged = sensor_dfs[0]
                print(f"  Starting with: {len(merged)} rows, {len(merged.columns)} columns")
                
                # Merge subsequent dataframes
                for i, df in enumerate(sensor_dfs[1:], 1):
                    before_rows = len(merged)
                    before_cols = len(merged.columns)
                    
                    merged = pd.merge(
                        merged, 
                        df, 
                        on='time_stamp', 
                        how='outer',  # Keep all timestamps from both files
                        suffixes=('', f'_dup{i}')
                    )
                    
                    after_rows = len(merged)
                    after_cols = len(merged.columns)
                    
                    print(f"  Step {i}: {before_rows} → {after_rows} rows, {before_cols} → {after_cols} cols")
                
                # Remove duplicate columns (keep first occurrence)
                duplicate_cols = [col for col in merged.columns if '_dup' in col]
                if duplicate_cols:
                    print(f"  Removing {len(duplicate_cols)} duplicate columns")
                    merged = merged.drop(columns=duplicate_cols)
                    print(f"  Final columns: {len(merged.columns)}")
        
        # STEP 4: Add metadata
        # Add and clean code
        merged['code'] = str(sensor_code).strip()
        
        # Map to site using the code_to_site mapping
        merged['site'] = merged['code'].map(code_to_site)
        
        mapped_count = merged['site'].notna().sum()
        if mapped_count == 0:
            print(f"  ⚠️  No site found for code {sensor_code}")
        else:
            site_name = merged['site'].dropna().iloc[0]
            print(f"  ✓ Mapped to site: {site_name} ({mapped_count} rows)")
        
        # Rename time_stamp to time_local for consistency
        if 'time_stamp' in merged.columns:
            merged = merged.rename(columns={'time_stamp': 'time_local'})
        
        # Show final key columns
        print(f"\n  Final merged data:")
        print(f"    Rows: {len(merged)}")
        key_cols = ['temperature', 'humidity', 'pm2.5_cf_1_a', 'pm2.5_cf_1_b', 'pm2.5_cf_1']
        for col in key_cols:
            if col in merged.columns:
                non_null = merged[col].notna().sum()
                print(f"    ✓ {col}: {non_null} non-null ({non_null/len(merged)*100:.1f}%)")
        
        all_data.append(merged)
    
    # STEP 5: Combine all sensors
    if all_data:
        print(f"\n{'='*70}")
        print("COMBINING ALL SENSORS")
        print(f"{'='*70}")
        combined = pd.concat(all_data, ignore_index=True)
        print(f"✓ Total: {len(combined)} rows from {len(all_data)} sensors")
        
        # Final check
        key_cols = ['temperature', 'humidity', 'pm2.5_cf_1_a', 'pm2.5_cf_1_b', 'pm2.5_cf_1']
        print(f"\nFinal column check:")
        for col in key_cols:
            if col in combined.columns:
                non_null = combined[col].notna().sum()
                print(f"  ✓ {col}: {non_null} non-null ({non_null/len(combined)*100:.1f}%)")
        
        return combined
    else:
        print("\n⚠️  No data processed")
        return pd.DataFrame()

In [50]:
def diagnose_timestamp_conversion(df):
    """
    Diagnose timestamp conversion issues in the pipeline
    """
    print("="*70)
    print("TIMESTAMP CONVERSION DIAGNOSTIC")
    print("="*70)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Total rows: {len(df)}")
    
    # Check what time columns exist
    print("\n--- Available Time Columns ---")
    time_cols = [col for col in df.columns if any(t in col.lower() for t in ['time', 'date', 'utc'])]
    for col in time_cols:
        print(f"  - {col}: {df[col].dtype}")
        non_null = df[col].notna().sum()
        print(f"    Non-null: {non_null} ({non_null/len(df)*100:.1f}%)")
        if non_null > 0:
            print(f"    Sample values: {df[col].dropna().head(3).tolist()}")
    
    # Test each conversion path
    print("\n" + "="*70)
    print("TESTING CONVERSION PATHS")
    print("="*70)
    
    # Path 1: UTCDateTime
    if 'UTCDateTime' in df.columns:
        print("\n✓ Path 1: UTCDateTime column found")
        print(f"  Original dtype: {df['UTCDateTime'].dtype}")
        print(f"  Sample original values:")
        print(f"    {df['UTCDateTime'].head(3).tolist()}")
        
        test_df = df.copy()
        test_df['UTCDateTime_converted'] = pd.to_datetime(test_df['UTCDateTime'], utc=True, errors='coerce')
        
        print(f"\n  After pd.to_datetime(utc=True):")
        print(f"    Dtype: {test_df['UTCDateTime_converted'].dtype}")
        print(f"    Non-null: {test_df['UTCDateTime_converted'].notna().sum()}")
        print(f"    Sample converted:")
        print(f"      {test_df['UTCDateTime_converted'].dropna().head(3).tolist()}")
        
        test_df['time_bsb_test'] = test_df['UTCDateTime_converted'].dt.tz_convert('America/Sao_Paulo')
        print(f"\n  After .tz_convert('America/Sao_Paulo'):")
        print(f"    Dtype: {test_df['time_bsb_test'].dtype}")
        print(f"    Non-null: {test_df['time_bsb_test'].notna().sum()}")
        print(f"    Sample BSB times:")
        print(f"      {test_df['time_bsb_test'].dropna().head(3).tolist()}")
    
    # Path 2: time_local
    elif 'time_local' in df.columns:
        print("\n✓ Path 2: time_local column found")
        print(f"  Original dtype: {df['time_local'].dtype}")
        print(f"  Sample original values:")
        print(f"    {df['time_local'].head(3).tolist()}")
        
        test_df = df.copy()
        test_df['time_local_converted'] = pd.to_datetime(test_df['time_local'], errors='coerce')
        
        print(f"\n  After pd.to_datetime():")
        print(f"    Dtype: {test_df['time_local_converted'].dtype}")
        print(f"    Has timezone: {test_df['time_local_converted'].dt.tz is not None}")
        print(f"    Non-null: {test_df['time_local_converted'].notna().sum()}")
        print(f"    Sample converted:")
        print(f"      {test_df['time_local_converted'].dropna().head(3).tolist()}")
        
        if test_df['time_local_converted'].dt.tz is None:
            test_df['time_bsb_test'] = test_df['time_local_converted'].dt.tz_localize('America/Sao_Paulo')
            print(f"\n  After .tz_localize('America/Sao_Paulo'):")
        else:
            test_df['time_bsb_test'] = test_df['time_local_converted'].dt.tz_convert('America/Sao_Paulo')
            print(f"\n  After .tz_convert('America/Sao_Paulo'):")
        
        print(f"    Dtype: {test_df['time_bsb_test'].dtype}")
        print(f"    Non-null: {test_df['time_bsb_test'].notna().sum()}")
        print(f"    Sample BSB times:")
        print(f"      {test_df['time_bsb_test'].dropna().head(3).tolist()}")
    
    # Path 3: time_stamp
    elif 'time_stamp' in df.columns:
        print("\n✓ Path 3: time_stamp column found")
        print(f"  Original dtype: {df['time_stamp'].dtype}")
        print(f"  Sample original values:")
        print(f"    {df['time_stamp'].head(3).tolist()}")
        
        test_df = df.copy()
        test_df['time_stamp_converted'] = pd.to_datetime(test_df['time_stamp'], utc=True, errors='coerce')
        
        print(f"\n  After pd.to_datetime(utc=True):")
        print(f"    Dtype: {test_df['time_stamp_converted'].dtype}")
        print(f"    Has timezone: {test_df['time_stamp_converted'].dt.tz is not None}")
        print(f"    Non-null: {test_df['time_stamp_converted'].notna().sum()}")
        print(f"    Sample converted:")
        print(f"      {test_df['time_stamp_converted'].dropna().head(3).tolist()}")
        
        if test_df['time_stamp_converted'].dt.tz is not None:
            test_df['time_bsb_test'] = test_df['time_stamp_converted'].dt.tz_convert('America/Sao_Paulo')
            print(f"\n  After .tz_convert('America/Sao_Paulo'):")
        else:
            test_df['time_bsb_test'] = test_df['time_stamp_converted'].dt.tz_localize('America/Sao_Paulo')
            print(f"\n  After .tz_localize('America/Sao_Paulo'):")
        
        print(f"    Dtype: {test_df['time_bsb_test'].dtype}")
        print(f"    Non-null: {test_df['time_bsb_test'].notna().sum()}")
        print(f"    Sample BSB times:")
        print(f"      {test_df['time_bsb_test'].dropna().head(3).tolist()}")
    else:
        print("\n❌ No recognized time column found!")
        print(f"Available columns: {df.columns.tolist()}")
    
    # Check if time_bsb already exists
    print("\n" + "="*70)
    print("CHECKING EXISTING time_bsb COLUMN")
    print("="*70)
    
    if 'time_bsb' in df.columns:
        print("\n✓ time_bsb column already exists")
        print(f"  Dtype: {df['time_bsb'].dtype}")
        print(f"  Non-null: {df['time_bsb'].notna().sum()} ({df['time_bsb'].notna().sum()/len(df)*100:.1f}%)")
        print(f"  Timezone: {df['time_bsb'].dt.tz if hasattr(df['time_bsb'].dt, 'tz') else 'N/A'}")
        print(f"  Sample values:")
        print(f"    {df['time_bsb'].dropna().head(5).tolist()}")
        
        # Check date range
        if df['time_bsb'].notna().sum() > 0:
            print(f"\n  Date range:")
            print(f"    Min: {df['time_bsb'].min()}")
            print(f"    Max: {df['time_bsb'].max()}")
    else:
        print("\n⚠️  time_bsb column does not exist yet")
    
    return df


In [51]:
def safe_read_csv_with_header_check(file):
    """Safely read CSV with encoding fallback"""
    dateparse = lambda x: datetime.strptime(x.replace('z', '+0000').replace('Z', '+0000'), "%Y/%m/%dT%H:%M:%S%z")
    
    try:
        df = pd.read_csv(
            file,
            encoding='ISO-8859-1',
            header=0,
            parse_dates=['UTCDateTime'],
            date_format=dateparse,
            on_bad_lines='skip'
        )
    except UnicodeDecodeError:
        df = pd.read_csv(
            file,
            encoding='latin1',
            header=0,
            parse_dates=['UTCDateTime'],
            date_format=dateparse,
            on_bad_lines='skip'
        )
    return df


def process_single_sd_folder(root_dir):
    """Process SD card data from a single folder"""
    df_list = []
    
    if not os.path.exists(root_dir):
        print(f"  ⚠ Folder not found: {root_dir}")
        return []
    
    for folder in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder)
        
        if os.path.isdir(folder_path):
            files = glob.glob(os.path.join(folder_path, "*.csv")) + glob.glob(os.path.join(folder_path, "*.xlsx"))
            
            for file in files:
                try:
                    if file.endswith(".csv"):
                        df = safe_read_csv_with_header_check(file)
                    else:
                        dateparse = lambda x: datetime.strptime(x.replace('z', '+0000').replace('Z', '+0000'), "%Y/%m/%dT%H:%M:%S%z")
                        df = pd.read_excel(file, parse_dates=['UTCDateTime'], date_format=dateparse)
                    
                    # PRIORITY 1: Use 'local' column if it exists and has non-null values
                    if 'local' in df.columns:
                        non_null_local = df['local'].notna().sum()
                        
                        if non_null_local > 0:
                            # Use 'local' column as the site
                            df['site'] = df['local']
                            # For rows where local is null, use folder name as fallback
                            df.loc[df['site'].isna(), 'site'] = folder
                            print(f"    ✓ {folder}/{os.path.basename(file)}: {len(df)} rows (using 'local' column for {non_null_local} rows)")
                        else:
                            # Local column exists but all null, use folder name
                            df['site'] = folder
                            print(f"    ✓ {folder}/{os.path.basename(file)}: {len(df)} rows (using folder name)")
                    else:
                        # PRIORITY 2: No 'local' column, use folder name
                        df['site'] = folder
                        print(f"    ✓ {folder}/{os.path.basename(file)}: {len(df)} rows (using folder name)")
                    
                    df_list.append(df)
                    
                except Exception as e:
                    print(f"    ✗ Failed: {folder}/{os.path.basename(file)}: {e}")
    
    return df_list


def process_all_sd_card_data(sd_folders):
    """Process all SD card data from multiple folders"""
    print("\n" + "="*70)
    print("PROCESSING SD CARD DATA")
    print("="*70)
    
    # Handle None or empty list
    if not sd_folders:
        print("⚠ No SD card folders configured - skipping")
        return pd.DataFrame()
    
    all_df_list = []
    
    for i, folder in enumerate(sd_folders, 1):
        print(f"\n[{i}/{len(sd_folders)}] Processing: {folder}")
        folder_dfs = process_single_sd_folder(folder)
        all_df_list.extend(folder_dfs)
    
    if all_df_list:
        combined_df = pd.concat(all_df_list, ignore_index=True)
        print(f"\n✓ SD card data combined: {len(combined_df)} total rows from {len(all_df_list)} files")
        
        # Show site distribution
        if 'site' in combined_df.columns:
            print(f"\nSites found:")
            site_counts = combined_df['site'].value_counts()
            for site, count in site_counts.head(10).items():
                print(f"  - {site}: {count} rows")
            if len(site_counts) > 10:
                print(f"  ... and {len(site_counts) - 10} more sites")
        
        return combined_df
    else:
        print("\n⚠ No SD card data found")
        return pd.DataFrame()

In [52]:

# ============================================================================
# EXTERNAL CSV PROCESSING (para_air type data)
# ============================================================================

def process_external_csv(external_csv_path):
    """
    Process external CSV data (e.g., para_air type data)
    """
    print("\n" + "="*70)
    print("PROCESSING EXTERNAL CSV")
    print("="*70)
    
    # Handle None or empty path
    if not external_csv_path:
        print("⚠ No external CSV configured - skipping")
        return pd.DataFrame()
    
    if not os.path.exists(external_csv_path):
        print(f"⚠ External CSV not found: {external_csv_path}")
        return pd.DataFrame()
    
    try:
        print(f"Loading: {external_csv_path}")
        df = pd.read_csv(external_csv_path, low_memory=False)
        print(f"✓ External CSV loaded: {len(df)} rows, {len(df.columns)} columns")
        
        # Show sample of columns
        print(f"  Columns: {list(df.columns[:10])}")
        if len(df.columns) > 10:
            print(f"  ... and {len(df.columns) - 10} more columns")
            
        # Rename mun_name to site
        if 'mun_name' in df.columns:
            df = df.rename(columns={'mun_name': 'site'})
            print(f"✓ Renamed 'mun_name' to 'site'")
        
        # Standardize datetime column
        if 'time_stamp' in df.columns:
            df['time_stamp'] = pd.to_datetime(df['time_stamp'])
            df['time_bsb'] = df['time_stamp']
            print(f"✓ Processed timestamp column")
        
        # Extract time components
        if 'time_bsb' in df.columns:
            df['year'] = df['time_bsb'].dt.year
            df['month'] = df['time_bsb'].dt.month
            df['day'] = df['time_bsb'].dt.day
            df['hour'] = df['time_bsb'].dt.hour
            df['time_bsb_copy'] = df['time_bsb']
        
        # Drop unnecessary columns
        columns_to_drop = ["mac_address", "firmware_ver", "hardware", "mem"]
        df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
        
        # Note: temperature and humidity columns should already have correct names
        # If they don't, they will be handled in the cleaning function
        
        print(f"✓ External CSV processed: {len(df)} rows")
        return df
        
    except Exception as e:
        print(f"✗ Error processing external CSV: {e}")
        return pd.DataFrame()


In [53]:

# ============================================================================
# DATA CLEANING AND CALIBRATION
# ============================================================================

import pandas as pd
import numpy as np

def clean_and_calibrate(df, config):
    """Apply cleaning filters and calibration corrections"""
    print("\n" + "="*70)
    print("CLEANING AND CALIBRATING DATA")
    print("="*70)
    
    if df.empty:
        print("⚠ No data to clean")
        return df
    
    initial_rows = len(df)
    print(f"Initial rows: {initial_rows}")
    
    # CRITICAL: Remove duplicate columns FIRST
    if len(df.columns) != len(set(df.columns)):
        print("⚠️  Duplicate columns detected, removing...")
        from collections import Counter
        dupes = {col: count for col, count in Counter(df.columns).items() if count > 1}
        print(f"   Duplicate columns: {dupes}")
        df = df.loc[:, ~df.columns.duplicated(keep='first')]
        print(f"   Columns after deduplication: {len(df.columns)}")
    
    # Convert datetime - handle multiple time column sources
    has_utcdatetime = 'UTCDateTime' in df.columns and df['UTCDateTime'].notna().sum() > 0
    has_time_local = 'time_local' in df.columns and df['time_local'].notna().sum() > 0
    has_time_stamp = 'time_stamp' in df.columns and df['time_stamp'].notna().sum() > 0
    has_time_bsb = 'time_bsb' in df.columns and df['time_bsb'].notna().sum() > 0
    
    print(f"Time columns found:")
    if has_utcdatetime:
        print(f"  - UTCDateTime: {df['UTCDateTime'].notna().sum()} rows")
    if has_time_local:
        print(f"  - time_local: {df['time_local'].notna().sum()} rows")
    if has_time_stamp:
        print(f"  - time_stamp: {df['time_stamp'].notna().sum()} rows")
    if has_time_bsb:
        print(f"  - time_bsb (existing): {df['time_bsb'].notna().sum()} rows")
    
    # Create time_bsb_new with proper timezone-aware dtype
    time_bsb_new = pd.Series([pd.NaT] * len(df), dtype='datetime64[ns, UTC]').dt.tz_convert('America/Sao_Paulo')
    df['time_bsb_new'] = time_bsb_new
    
    # Process UTCDateTime
    if has_utcdatetime:
        utc_converted = pd.to_datetime(df['UTCDateTime'], utc=True, errors='coerce')
        utc_bsb = utc_converted.dt.tz_convert('America/Sao_Paulo')
        df.loc[utc_converted.notna(), 'time_bsb_new'] = utc_bsb[utc_converted.notna()].values
        print(f"  → Converted {utc_converted.notna().sum()} rows from UTCDateTime")
    
    # Process time_local
    if has_time_local:
        time_local_mask = df['time_bsb_new'].isna() & df['time_local'].notna()
        if time_local_mask.sum() > 0:
            if df['time_local'].dt.tz is None:
                df.loc[time_local_mask, 'time_bsb_new'] = df.loc[time_local_mask, 'time_local'].dt.tz_localize('America/Sao_Paulo').values
            else:
                df.loc[time_local_mask, 'time_bsb_new'] = df.loc[time_local_mask, 'time_local'].dt.tz_convert('America/Sao_Paulo').values
            print(f"  → Converted {time_local_mask.sum()} rows from time_local")
    
    # Process time_stamp
    if has_time_stamp:
        time_stamp_mask = df['time_bsb_new'].isna() & df['time_stamp'].notna()
        if time_stamp_mask.sum() > 0:
            timestamp_converted = pd.to_datetime(df.loc[time_stamp_mask, 'time_stamp'], utc=True, errors='coerce')
            if timestamp_converted.dt.tz is not None:
                df.loc[time_stamp_mask, 'time_bsb_new'] = timestamp_converted.dt.tz_convert('America/Sao_Paulo').values
            else:
                df.loc[time_stamp_mask, 'time_bsb_new'] = timestamp_converted.dt.tz_localize('America/Sao_Paulo').values
            print(f"  → Converted {time_stamp_mask.sum()} rows from time_stamp")
    
    # Handle existing time_bsb
    if has_time_bsb:
        existing_bsb_mask = df['time_bsb_new'].isna() & df['time_bsb'].notna()
        if existing_bsb_mask.sum() > 0:
            existing_converted = pd.to_datetime(df.loc[existing_bsb_mask, 'time_bsb'], errors='coerce')
            if existing_converted.dt.tz is None and existing_converted.notna().sum() > 0:
                df.loc[existing_bsb_mask, 'time_bsb_new'] = existing_converted.dt.tz_localize('America/Sao_Paulo').values
            elif existing_converted.notna().sum() > 0:
                df.loc[existing_bsb_mask, 'time_bsb_new'] = existing_converted.dt.tz_convert('America/Sao_Paulo').values
            print(f"  → Converted {existing_bsb_mask.sum()} rows from existing time_bsb")
    
    # Replace old time_bsb with new one
    df['time_bsb'] = pd.to_datetime(df['time_bsb_new'], utc=True).dt.tz_convert('America/Sao_Paulo')
    df = df.drop(columns=['time_bsb_new'])
    
    # Check results
    final_count = df['time_bsb'].notna().sum()
    print(f"\n✓ Final time_bsb: {final_count} rows ({final_count/len(df)*100:.1f}%)")
    print(f"  Dtype: {df['time_bsb'].dtype}")
    
    if final_count == 0:
        print("❌ No valid timestamps created!")
        return df
    
    # Extract time components
    df['year'] = df['time_bsb'].dt.year
    df['month'] = df['time_bsb'].dt.month
    df['day'] = df['time_bsb'].dt.day
    df['hour'] = df['time_bsb'].dt.hour
    df['time_bsb_copy'] = df['time_bsb']
    df['site'] = df['site'].map(site_name_map).fillna(df['site']) 

    # TEMPERATURE FILTER
    if 'temperature' not in df.columns:
        if 'current_temp_f' in df.columns:
            df['temperature'] = df['current_temp_f']
        else:
            print("⚠️  No temperature column found!")
            return df
    
    print(f"Using temperature column")
    
    if isinstance(df['temperature'], pd.DataFrame):
        print(f"  ⚠️ 'temperature' is a DataFrame, using first")
        df['temperature'] = df['temperature'].iloc[:, 0]
    
    df['temperature'] = pd.to_numeric(df['temperature'], errors='coerce')
    
    if len(df.columns) != len(set(df.columns)):
        df = df.loc[:, ~df.columns.duplicated(keep='first')]
    
    before_temp = len(df)
    df = df[df['temperature'].notna()]
    print(f"After removing null temperatures: {len(df)} rows ({before_temp - len(df)} removed)")
    
    before_filter = len(df)
    df = df[(df['temperature'] >= config['min_temp']) & (df['temperature'] <= config['max_temp'])]
    print(f"After temperature filter: {len(df)} rows ({before_filter - len(df)} removed)")
    
    # HUMIDITY FILTER
    if 'humidity' not in df.columns:
        if 'current_humidity' in df.columns:
            df['humidity'] = df['current_humidity']
        else:
            print("⚠️  No humidity column found!")
            return df
    
    print(f"Using humidity column")
    
    if isinstance(df['humidity'], pd.DataFrame):
        print(f"  ⚠️ 'humidity' is a DataFrame, using first")
        df['humidity'] = df['humidity'].iloc[:, 0]
    
    df['humidity'] = pd.to_numeric(df['humidity'], errors='coerce')
    
    if len(df.columns) != len(set(df.columns)):
        df = df.loc[:, ~df.columns.duplicated(keep='first')]
    
    before_humidity = len(df)
    df = df[df['humidity'].notna()]
    print(f"After removing null humidity: {len(df)} rows ({before_humidity - len(df)} removed)")
    
    before_filter = len(df)
    df = df[(df['humidity'] >= config['min_humidity']) & (df['humidity'] <= config['max_humidity'])]
    print(f"After humidity filter: {len(df)} rows ({before_filter - len(df)} removed)")
    
    # PM2.5 CHANNEL PROCESSING
    for col in ['pm2.5_cf_1_a', 'pm2.5_cf_1_b']:
        if col in df.columns:
            if isinstance(df[col], pd.DataFrame):
                print(f"⚠️  Column '{col}' is a DataFrame, converting to Series")
                df[col] = df[col].iloc[:, 0]
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Calculate channel agreement
    if 'pm2.5_cf_1_a' in df.columns and 'pm2.5_cf_1_b' in df.columns:
        df['pm2.5_cf_1_a'] = pd.to_numeric(df['pm2.5_cf_1_a'], errors='coerce')
        df['pm2.5_cf_1_b'] = pd.to_numeric(df['pm2.5_cf_1_b'], errors='coerce')
        
        df['abs_diff'] = (df['pm2.5_cf_1_a'] - df['pm2.5_cf_1_b']).abs()
        df['percent_diff'] = 100 * df['abs_diff'] / df[['pm2.5_cf_1_a', 'pm2.5_cf_1_b']].mean(axis=1)
        
        df['channel_agreement'] = (df['abs_diff'] < config['max_abs_diff']) | (df['percent_diff'] < config['max_percent_diff'])
        
        agreement_count = df['channel_agreement'].sum()
        disagreement_count = (~df['channel_agreement']).sum()
        print(f"Channel agreement: {agreement_count} rows agree, {disagreement_count} rows disagree")
    else:
        print(f"⚠ Warning: Missing channel B data")
        if 'pm2.5_cf_1_a' in df.columns:
            df['pm2.5_cf_1_a'] = pd.to_numeric(df['pm2.5_cf_1_a'], errors='coerce')
            df['channel_agreement'] = True
    
    # Apply calibration corrections ONLY to rows in agreement
    if 'pm2.5_cf_1_a' in df.columns and 'channel_agreement' in df.columns:
        df['PM2.5_US_correction'] = np.nan
        df['PM2.5_quadratic'] = np.nan
        df['PM2.5_transitional_correction'] = np.nan
        df['PM2_5_corrected'] = np.nan
        
        agree_mask = df['channel_agreement'] == True
        
        if agree_mask.sum() > 0:
            df.loc[agree_mask, 'PM2.5_US_correction'] = (
                df.loc[agree_mask, 'pm2.5_cf_1_a'] * 0.524 - 
                0.0862 * df.loc[agree_mask, 'humidity'] + 5.75
            )
            
            df.loc[agree_mask, 'PM2.5_quadratic'] = (
                df.loc[agree_mask, 'pm2.5_cf_1_a']**2 * 4.21 * 10**-4 + 
                df.loc[agree_mask, 'pm2.5_cf_1_a'] * 0.392 + 3.44
            )
            
            df.loc[agree_mask, 'PM2.5_transitional_correction'] = (
                (0.0244 * df.loc[agree_mask, 'pm2.5_cf_1_a'] - 13.9) * df.loc[agree_mask, 'PM2.5_quadratic'] +
                (1 - (0.0244 * df.loc[agree_mask, 'pm2.5_cf_1_a'] - 13.9)) * df.loc[agree_mask, 'PM2.5_US_correction']
            )
            
            pm_values = df.loc[agree_mask, 'pm2.5_cf_1_a']
            df.loc[agree_mask, 'PM2_5_corrected'] = np.select(
                [
                    pm_values < 570,
                    (pm_values >= 570) & (pm_values < 611),
                    pm_values >= 611
                ],
                [
                    df.loc[agree_mask, 'PM2.5_US_correction'],
                    df.loc[agree_mask, 'PM2.5_transitional_correction'],
                    df.loc[agree_mask, 'PM2.5_quadratic']
                ],
                default=np.nan
            )
            
            calibrated_count = df['PM2_5_corrected'].notna().sum()
            print(f"✓ Calibration corrections applied to {calibrated_count} rows in agreement")
    
    print(f"\nFinal: {len(df)} rows ({initial_rows - len(df)} total removed, {len(df)/initial_rows*100:.1f}% retained)")
    
    return df
# ============================================================================
# MAIN PIPELINE
# ============================================================================






In [54]:
def standardize_columns(df, source_type):
    """Standardize column names across different data sources."""
    df = df.copy()
    if len(df.columns) != len(set(df.columns)):
        print(f"  ⚠ Removing duplicate columns in {source_type}")
        df = df.loc[:, ~df.columns.duplicated()]
    
    # Step 2: Rename columns using your map
    rename_map = {
        #'pm2_5_cf_1': 'pm2.5_cf_1_a',
        'pm2_5_cf_1_a': 'pm2.5_cf_1_a',
        'pm2_5_cf_1_b': 'pm2.5_cf_1_b',
        'pm25_cf_1_a': 'pm2.5_cf_1_a',
        'pm25_cf_1_b': 'pm2.5_cf_1_b',
        'current_humidity': 'humidity',
        'current_temp_f': 'temperature',
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    
    # Step 3: Remove duplicates again after renaming (in case rename created them)
    if len(df.columns) != len(set(df.columns)):
        print(f"  ⚠ Rename created duplicate columns, keeping last occurrence")
        df = df.loc[:, ~df.columns.duplicated(keep='last')]
     # Step 2: CRITICAL - Duplicate single channel PM2.5 to both A and B
    # This is for download data that only has one sensor
     
    print(f"  Standardized {source_type}: {len(df)} rows")
    return df


In [55]:
def run_pipeline(config):
    """Execute the complete data pipeline"""
    print("\n" + "="*70)
    print("PM2.5 DATA PIPELINE")
    print("="*70)
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Step 1: Backup existing data
    #create_backup(config['master_csv'], config['backup_folder'])
    
    # Step 2: Load existing master data
    master_df = load_master_data(config['master_csv'])
    
    # Step 3: Process all SD card folders
    print("\n--- Processing SD Card Data ---")
    sd_data = process_all_sd_card_data(config['sd_card_folders'])
    # CRITICAL: Standardize BEFORE combining
    if not sd_data.empty:
        print(f"Before standardization: {len(sd_data)} rows")
        sd_data = standardize_columns(sd_data, "SD card data")
        print(f"After standardization: {len(sd_data)} rows")
        
        # Verify standardization worked
        key_cols = ['temperature', 'humidity', 'pm2.5_cf_1_a', 'pm2.5_cf_1_b']
        print(f"Key columns after standardization:")
        for col in key_cols:
            if col in sd_data.columns:
                non_null = sd_data[col].notna().sum()
                print(f"  ✓ {col}: {non_null} non-null ({non_null/len(sd_data)*100:.1f}%)")
            else:
                print(f"  ✗ {col}: MISSING")
    
    # Step 4: Process all download folders
    print("\n--- Processing Download Data ---")
    download_data = process_all_download_data(
        config['download_folders'],
        config['reference_file'],
        config['code_column'],
        config['site_column']
    )
    
    # CRITICAL: Standardize BEFORE combining
    if not download_data.empty:
        print(f"Before standardization: {len(download_data)} rows")
        download_data = standardize_columns(download_data, "download data")
        print(f"After standardization: {len(download_data)} rows")
        
        # Verify standardization worked
        key_cols = ['temperature', 'humidity', 'pm2.5_cf_1_a', 'pm2.5_cf_1_b']
        print(f"Key columns after standardization:")
        for col in key_cols:
            if col in download_data.columns:
                non_null = download_data[col].notna().sum()
                print(f"  ✓ {col}: {non_null} non-null ({non_null/len(download_data)*100:.1f}%)")
            else:
                print(f"  ✗ {col}: MISSING")
    # Step 5: Process external CSV (para_air type data)
    print(config.get('external_csv'))
    external_data = process_external_csv(config.get('external_csv'))
    external_data = standardize_columns(external_data, "external_csv")

    # Step 6: Combine all new data
    print("\n" + "="*70)
    print("COMBINING NEW DATA")
    print("="*70)
    
    new_data_list = [df for df in [sd_data, download_data, external_data] if not df.empty]
    
    if new_data_list:
        new_combined = pd.concat(new_data_list, ignore_index=True)
        print(f"✓ Combined new data: {len(new_combined)} rows")
        print(f"  - SD card data: {len(sd_data) if not sd_data.empty else 0} rows")
        print(f"  - Download data: {len(download_data) if not download_data.empty else 0} rows")
        print(f"  - External CSV data: {len(external_data) if not external_data.empty else 0} rows")
    else:
        print("⚠ No new data to process")
        return master_df
    #new_combined=diagnose_timestamp_conversion(new_combined)
    
    # Step 7: Clean and calibrate new data
    new_cleaned = clean_and_calibrate(new_combined, config)
    
    # Step 8: Merge with master data
    print("\n" + "="*70)
    print("MERGING WITH MASTER DATA")
    print("="*70)
    
    if not master_df.empty:
        # Remove duplicates based on timestamp and site
        all_data = pd.concat([master_df, new_cleaned], ignore_index=True)
        initial_count = len(all_data)
        
        if 'time_bsb_copy' in all_data.columns and 'site' in all_data.columns:
            all_data = all_data.drop_duplicates(subset=['time_bsb_copy', 'site'], keep='last')
            duplicates_removed = initial_count - len(all_data)
            print(f"✓ Removed {duplicates_removed} duplicate records")
        
        final_df = all_data
    else:
        final_df = new_cleaned
    
    # Step 8: Save to master CSV
    print("SAVING RESULTS")
    print("="*70)
    
    #os.makedirs(os.path.dirname(config['master_csv']), exist_ok=True)
    final_df.to_csv(config['master_csv'], index=False)
    print(f"✓ Master CSV saved: {config['master_csv']}")
    print(f"  Total rows: {len(final_df)}")
    print(f"  New rows added: {len(new_cleaned)}")
    if not master_df.empty:
        print(f"  Duplicates removed: {duplicates_removed}")
    
    # Summary statistics
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"Sites: {final_df['site'].nunique()}")
    if 'time_bsb_copy' in final_df.columns:
        print(f"Date range: {final_df['time_bsb_copy'].min()} to {final_df['time_bsb_copy'].max()}")
    print(f"\nTop 5 sites by record count:")
    print(final_df['site'].value_counts().head())
    
    print(f"\n✓ Pipeline completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    return final_df

In [56]:
# ============================================================================
# EXECUTE PIPELINE
# ============================================================================

if __name__ == "__main__":
    result_df = run_pipeline(CONFIG)
    print("\n" + "="*70)
    print("PIPELINE COMPLETE")
    print("="*70)


PM2.5 DATA PIPELINE
Started: 2026-01-05 09:59:53

ℹ No master data found - starting fresh

--- Processing SD Card Data ---

PROCESSING SD CARD DATA

[1/2] Processing: /Users/aatwood/Downloads/Sensores dados 2024-2025
    ✓ Ar_Uaça_TI/20241102.csv: 568 rows (using folder name)
    ✓ Ar_Uaça_TI/20241116.csv: 1 rows (using folder name)
    ✓ Ar_Uaça_TI/20250219.csv: 286 rows (using folder name)
    ✓ Ar_Uaça_TI/20250225.csv: 339 rows (using folder name)
    ✓ Ar_Uaça_TI/20240817.csv: 374 rows (using folder name)
    ✓ Ar_Uaça_TI/20240803.csv: 536 rows (using folder name)
    ✓ Ar_Uaça_TI/20240618.csv: 446 rows (using folder name)
    ✓ Ar_Uaça_TI/20240630.csv: 414 rows (using folder name)
    ✓ Ar_Uaça_TI/20240624.csv: 335 rows (using folder name)
    ✓ Ar_Uaça_TI/20240625.csv: 387 rows (using folder name)
    ✓ Ar_Uaça_TI/20240619.csv: 406 rows (using folder name)
    ✓ Ar_Uaça_TI/20240802.csv: 171 rows (using folder name)
    ✓ Ar_Uaça_TI/20240816.csv: 412 rows (using fold

/var/folders/t7/y1wr1p2d33s27zhc0snt07rh0000gp/T/ipykernel_3988/1681392197.py:6: DtypeWarning: Columns (2,4,9,10,11,12,13,14,15,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    ✓ all_data/ipam_purple_air.csv: 4552339 rows (using 'local' column for 4552339 rows)

✓ SD card data combined: 5953261 total rows from 2379 files

Sites found:
  - Ar_Canarana_URB: 508752 rows
  - Ar_Altamira_URB: 440517 rows
  - Ar_Capitao_Poco_URB: 422772 rows
  - Ar_Chapada_dos_Guimaraes_UC: 413830 rows
  - Ar_Sinop_URB: 409870 rows
  - Ar_Cunia_UC: 409610 rows
  - Ar_Mamiraua_UC: 358044 rows
  - AR_Alto_Turiacu_TI: 357596 rows
  - Ar_Tanguro_Fazenda: 261225 rows
  - Ar_Alta_Floresta_URB: 249255 rows
  ... and 17 more sites
Before standardization: 5953261 rows
  Standardized SD card data: 5953261 rows
After standardization: 5953261 rows
Key columns after standardization:
  ✓ temperature: 5952852 non-null (100.0%)
  ✓ humidity: 5952852 non-null (100.0%)
  ✗ pm2.5_cf_1_a: MISSING
  ✓ pm2.5_cf_1_b: 5952842 non-null (100.0%)

--- Processing Download Data ---

PROCESSING DOWNLOAD DATA (WITH CROSS-FOLDER MERGE)
✓ Reference file loaded: 51 rows
✓ Created 57 code-to-site mappings

--- S

/var/folders/t7/y1wr1p2d33s27zhc0snt07rh0000gp/T/ipykernel_3988/2041790734.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['2024-11-02T00:01:19.000000000' '2024-11-02T00:03:18.000000000'
 '2024-11-02T00:05:18.000000000' ... '2023-11-29T13:18:20.000000000'
 '2023-11-29T13:20:20.000000000' '2023-11-29T13:22:20.000000000']' has dtype incompatible with datetime64[ns, America/Sao_Paulo], please explicitly cast to a compatible dtype first.
  df.loc[utc_converted.notna(), 'time_bsb_new'] = utc_bsb[utc_converted.notna()].values


  → Converted 5952790 rows from UTCDateTime
  → Converted 14397472 rows from time_local
  → Converted 2 rows from existing time_bsb

✓ Final time_bsb: 20350264 rows (100.0%)
  Dtype: datetime64[ns, America/Sao_Paulo]
Using temperature column
After removing null temperatures: 19641993 rows (708740 removed)
After temperature filter: 19641981 rows (12 removed)
Using humidity column
After removing null humidity: 19641970 rows (11 removed)
After humidity filter: 19641968 rows (2 removed)
Channel agreement: 11984218 rows agree, 7657750 rows disagree
✓ Calibration corrections applied to 11984218 rows in agreement

Final: 19641968 rows (708765 total removed, 96.5% retained)

MERGING WITH MASTER DATA
SAVING RESULTS
✓ Master CSV saved: /Users/aatwood/Documents/PM25_master_data.csv
  Total rows: 19641968
  New rows added: 19641968

SUMMARY
Sites: 50
Date range: 2022-12-31 21:00:05-03:00 to 2025-12-30 20:59:58-03:00

Top 5 sites by record count:
site
Canarana URB           1283676
Santarém URB   

In [18]:
result_df.columns

Index(['time_stamp', 'sensor_index', 'private', 'rssi', 'uptime', 'pa_latency',
       'memory', 'latitude', 'longitude', 'altitude', 'humidity', 'humidity_a',
       'humidity_b', 'temperature', 'temperature_a', 'temperature_b',
       'pressure', 'pressure_a', 'pressure_b', 'voc', 'voc_a', 'voc_b',
       'analog_input', 'pm2_5_alt', 'pm2_5_alt_a', 'pm2_5_alt_b',
       '0_3_um_count', '0_3_um_count_a', '0_3_um_count_b', '0_5_um_count',
       '0_5_um_count_a', '0_5_um_count_b', '1_0_um_count', '1_0_um_count_a',
       '1_0_um_count_b', '2_5_um_count', '2_5_um_count_a', '2_5_um_count_b',
       '5_0_um_count', '5_0_um_count_a', '5_0_um_count_b', '10_0_um_count',
       '10_0_um_count_a', '10_0_um_count_b', 'pm1_0_cf_1', 'pm1_0_cf_1_a',
       'pm1_0_cf_1_b', 'pm1_0_atm', 'pm1_0_atm_a', 'pm1_0_atm_b', 'pm2_5_atm',
       'pm2_5_atm_a', 'pm2_5_atm_b', 'pm2.5_cf_1_a', 'pm2_5_cf_1_a',
       'pm2.5_cf_1_b', 'pm10_0_atm', 'pm10_0_atm_a', 'pm10_0_atm_b',
       'pm10_0_cf_1', 'pm10_0_cf_1_